In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rpa_control.models import ABModel, mu, a1, b1, b1_on, b2
from rpa_control import simulation

# AB system parameter sweep

Helper side-notebook for exploring how each parameter of the AB homeostatic pair (see `figure4 copy.ipynb`, `SI_strategies_with_clonal_selection.ipynb`) shapes the response to **one on/off treatment cycle** -- dip depth, recovery speed, over- vs under-damped return to baseline, etc. No tumor variable here, just `A`, `B`.

$$\dot A = a_1(\mu - B), \qquad \dot B = b_1(t)\,A - b_2 B$$

Base parameters (`rpa_control.models`): $\mu=1$, $a_1=1$, $b_1=1$ (off), $b_{1,on}=0.5$ (on), $b_2=2$.

Each section below tweaks **one parameter at a time** (0.5x, 1x, 2x the base value), holding everything else fixed. Simulation still goes through `simulation.simulate_rpa_on_off` (with `plot=False`), but instead of its default per-run figure, all three variants (plus the base case) are overlaid on one pair of axes ($A$, $B$) so the effect of the tweak is directly visible. The base run is always drawn as a dashed black line for reference.

In [ ]:
T_ON, T_OFF = 10.0, 15.0     # fixed cycle used for every comparison below
MULTIPLIERS = [0.5, 1.0, 2.0]

BASE = dict(a1=a1, b2=b2, mu=mu, b1=b1)

def _concat_on_off(sol_on, sol_off, t_on):
    """Stitch the on/off phases of a simulate_rpa_on_off(..., plot=False) result
    into single (t, A, B) arrays."""
    t = np.concatenate([sol_on.t, sol_off.t + t_on])
    A = np.concatenate([sol_on.y[0], sol_off.y[0]])
    B = np.concatenate([sol_on.y[1], sol_off.y[1]])
    return t, A, B

def run_variant(param, mult):
    """Build an ABModel with `param` scaled by `mult` (others held at BASE),
    simulate one on/off cycle from its own off-treatment steady state.
    Returns (t, A, B) concatenated across the on+off phases."""
    params = dict(BASE)
    params[param] = BASE[param] * mult
    model = ABModel(**params)
    y0 = model.steady_state()
    sol_on, sol_off = simulation.simulate_rpa_on_off(
        model, 'b1', b1_on, params['b1'], T_ON, T_OFF, y0=y0, plot=False)
    return _concat_on_off(sol_on, sol_off, T_ON)

def plot_ab_overlay(runs, title=''):
    """runs: list of (label, t, A, B). The run labeled 'base' is drawn as a
    dashed black reference line; every run is overlaid on one pair of axes."""
    fig, (ax_A, ax_B) = plt.subplots(2, 1, figsize=(8, 5), sharex=True,
                                     gridspec_kw={'hspace': 0.08})
    for ax in (ax_A, ax_B):
        ax.axvspan(0, T_ON, color='lightgray', alpha=0.4, lw=0)
        sns.despine(ax=ax)

    color_i = 0
    for label, t, A, B in runs:
        if label == 'base':
            style = dict(color='k', lw=2.2, ls='--', zorder=5)
        else:
            style = dict(color=f'C{color_i}', lw=1.6)
            color_i += 1
        ax_A.plot(t, A, label=label, **style)
        ax_B.plot(t, B, label=label, **style)

    ax_A.set_ylabel('$A$', fontsize=10)
    ax_B.set_ylabel('$B$', fontsize=10)
    ax_B.set_xlabel('time', fontsize=10)
    ax_A.legend(fontsize=8, frameon=False, loc='upper right')
    ax_A.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

## Tweak $\mu$ (homeostatic set-point)

In [ ]:
runs = [(f'mu={mu*mult:.3g} ({mult:g}x)' if mult != 1.0 else 'base', *run_variant('mu', mult))
       for mult in MULTIPLIERS]
plot_ab_overlay(runs, title=f'Tweaking mu (base={mu:.3g})')

## Tweak $a_1$ (adaptation rate: how fast $A$ reacts to the $B$ error)

In [ ]:
runs = [(f'a1={a1*mult:.3g} ({mult:g}x)' if mult != 1.0 else 'base', *run_variant('a1', mult))
       for mult in MULTIPLIERS]
plot_ab_overlay(runs, title=f'Tweaking a1 (base={a1:.3g})')

## Tweak $b_2$ ($B$'s own decay rate)

In [ ]:
runs = [(f'b2={b2*mult:.3g} ({mult:g}x)' if mult != 1.0 else 'base', *run_variant('b2', mult))
       for mult in MULTIPLIERS]
plot_ab_overlay(runs, title=f'Tweaking b2 (base={b2:.3g})')

## Tweak $b_1$ (off-treatment reaction rate -- shifts the pre-treatment baseline itself, since $A^*=b_2\mu/b_1$)

In [ ]:
runs = [(f'b1={b1*mult:.3g} ({mult:g}x)' if mult != 1.0 else 'base', *run_variant('b1', mult))
       for mult in MULTIPLIERS]
plot_ab_overlay(runs, title=f'Tweaking b1 (base={b1:.3g})')

## Tweak $b_{1,on}$ (treatment depth -- how strongly the drug suppresses $b_1$ while on)

This one isn't a model attribute -- it's the `on_value` passed straight to `simulate_rpa_on_off` -- so the loop looks slightly different: the model (hence $A(0), B(0)$) stays at baseline, only the on-phase value changes.

In [ ]:
model_base = ABModel(**BASE)
y0_base = model_base.steady_state()

runs = []
for mult in MULTIPLIERS:
    b1_on_variant = b1_on * mult
    label = f'b1_on={b1_on_variant:.3g} ({mult:g}x)' if mult != 1.0 else 'base'
    sol_on, sol_off = simulation.simulate_rpa_on_off(
        model_base, 'b1', b1_on_variant, b1, T_ON, T_OFF, y0=y0_base, plot=False)
    runs.append((label, *_concat_on_off(sol_on, sol_off, T_ON)))

plot_ab_overlay(runs, title=f'Tweaking b1_on (base={b1_on:.3g})')